# Notebook 00: Preparación y limpieza de datos - Planta fotovoltaica LECA1

## Descripción
Este notebook ejecuta el **pipeline completo de preparación y limpieza** de los datos
de radiación y producción de la planta LECA1, desde los archivos CSV brutos hasta un
dataset listo para unificiar con otros datasets y proceder al entrenamiento de modelos en notebooks posteriores.

## Flujo del notebook
1. Importar librerías y definir variables globales
2. Unificar los CSV brutos en un único DataFrame
3. Rellenar datos de radiación faltantes con estaciones de Weather Underground
4. Filtrar el rango temporal válido
5. Limpiar nulos, negativos y días vacíos
6. Corregir radiación nocturna e inconsistencias sensor/producción
7. Rellenar huecos temporales
8. Detectar y corregir outliers
9. Añadir temperatura ambiente (Open-Meteo / Weather Underground)
10. Guardar el CSV final limpio

**Este notebook solo prepara y valida el dataset. No entrena modelos.**

## Puntos clave
- Resolución temporal de **15 minutos**, periodo **enero 2022 – mayo 2024**.
- Se descarta la producción desde junio 2024 por operar la instalación en modo de regulación.
- El relleno de radiación se realiza iterando sobre un pool de estaciones WU ordenadas por correlación estadística.
- El CSV de salida es la entrada del notebook de análisis y entrenamiento.


### 1. Importación de librerías


In [ ]:
import os
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import plotly.io as pio

from utils.Preparacion_Dataset_15min_LECA1 import crear_dataframe
from utils.fill_radiation_data_LECA1 import Rellenar_datos_radiacion_from_wu
from utils.Analisis_datos_LECA1 import (
    cargar_y_filtrar_datos,
    graficar_matplotlib_estatico,
    limpiar_nulos_y_negativos,
    eliminar_dias_vacios,
    corregir_radiacion_nocturna,
    comprobar_radiacion_constante,
    corregir_potencia_sin_radiacion,
    procesar_huecos_temporales,
    detectar_y_corregir_outliers,
    generar_reporte_interactivo,
    analizar_correlacion_mensual,
)
from utils.add_wunderground_temperature import (
    get_wunderground_history,
    process_and_merge,
)
from utils.add_openmeteo_temperature import add_temperature_from_openmeteo

warnings.filterwarnings("ignore", category=FutureWarning)
pio.renderers.default = "notebook"


### 2. Variables globales
Se definen las rutas de entrada y salida mediante rutas relativas a la raíz del
repositorio, el rango temporal inicial y el flag .


In [ ]:
# Raíz del repositorio (la carpeta que contiene notebooks/, utils/, data/...)
PROJECT_ROOT = Path(".").resolve()

# Directorio con los CSV brutos de LECA1
# Ajusta esta ruta relativa a la estructura local de datos si es necesario.
INPUT_FOLDER = PROJECT_ROOT / "data" / "raw" / "LECA1"

# Directorio de resultados intermedios y finales
RESULTS_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CSV_MERGED           = RESULTS_DIR / "Datos_LECA1_15min.csv"
CSV_RADIATION_FILLED = RESULTS_DIR / "Datos_LECA1_Rellenos.csv"
CSV_CLEAN            = RESULTS_DIR / "Datos_LECA1_Limpio.csv"

# Directorio de cache de Weather Underground
WU_CACHE_DIR = PROJECT_ROOT / "data" / "external" / "wunderground"

# Rango temporal inicial (se ajusta más adelante tras el análisis visual)
FECHA_INICIO = "2022-01-01"
FECHA_FIN    = "2025-01-31"

# Controla si se ejecuta el relleno de huecos temporales
RELLENAR_HUECOS = True


### 3. Unificación de archivos CSV en un único DataFrame
 Recorre las carpetas de años y meses buscando archivos y
ejecuta el flujo completo de preparación:

1. **Iteración:** Recorre subcarpetas de año y mes buscando archivos .
2. **Normalización:** Selecciona las columnas de radiación y potencia y las renombra al estándar del proyecto.
3. **Limpieza:** Convierte textos a numérico, gestiona errores de codificación y elimina valores negativos.
4. **Enriquecimiento:** Genera variables derivadas del tiempo (hora, mes, día del año) y sus codificaciones cíclicas (seno/coseno).
5. **Exportación:** Guarda el histórico consolidado y ordenado cronológicamente.

La ejecución es condicional: si el archivo ya existe no se regenera.


In [ ]:
if not CSV_MERGED.exists():
    crear_dataframe(INPUT_FOLDER, CSV_MERGED)
else:
    print(f"Archivo ya existente: {CSV_MERGED}")


### 4. Relleno de datos faltantes de radiación con Weather Underground
 Ejecuta un pipeline iterativo para cubrir los
huecos de radiación:

1. **Inicialización:** Crea las carpetas necesarias y carga el CSV con los datos originales.
2. **Detección de anomalías:** Identifica días con radiación constante (sensor bloqueado) y los resetea a NaN.
3. **Ranking de estaciones:** Ordena las estaciones del pool por correlación con los datos de la planta.
4. **Contexto solar:** Calcula amanecer y anochecer por día para distinguir ceros nocturnos legítimos de huecos reales.
5. **Bucle de relleno:** Para cada estación del ranking, recupera los datos (caché o descarga) y rellena los huecos con una tolerancia temporal de ±15 min.
6. **Guardado:** Elimina columnas auxiliares y exporta el CSV con los huecos rellenos.

La ejecución es condicional: si el archivo ya existe no se regenera.


In [ ]:
if not CSV_RADIATION_FILLED.exists():
    Rellenar_datos_radiacion_from_wu(
        input_csv=str(CSV_MERGED),
        output_folder=str(RESULTS_DIR),
        wunderground_folder=str(WU_CACHE_DIR),
        correlation_folder=str(PROJECT_ROOT / "data" / "external" / "correlations"),
        stations_pool=[
            "ICUBOD2", "ILOSRB1", "IALMAZ2",
            "IQUINT63", "IGOLMA6", "ISORIA35",
        ],
        sample_year=2024,
        tolerance="15min",
        min_overlap_points=48,
    )
else:
    print(f"Archivo ya existente: {CSV_RADIATION_FILLED}")


### 5. Filtrado del rango temporal válido
Se carga el dataset con los huecos rellenos y se representa sin filtros para
inspección visual. Del análisis se desprenden las siguientes incidencias:

- Datos de radiación incompletos durante parte de 2022, no recuperables con Weather Underground.
- Producción anómalamente baja desde junio–julio 2024.
- La instalación opera en modo de regulación desde final de junio 2024, lo que rompe la correlación habitual entre radiación y producción.

Por tanto, el periodo de análisis queda acotado a **enero 2022 – mayo 2024**.


In [ ]:
# Visualización sin filtro temporal para inspección inicial
df_raw = cargar_y_filtrar_datos(CSV_RADIATION_FILLED, FECHA_INICIO, FECHA_FIN, filtrar_datos=False)
graficar_matplotlib_estatico(df_raw, [])


In [ ]:
# Recorte al periodo válido
FECHA_INICIO = "2022-01-01"
FECHA_FIN    = "2024-05-31"

df = cargar_y_filtrar_datos(CSV_RADIATION_FILLED, FECHA_INICIO, FECHA_FIN, filtrar_datos=True)
graficar_matplotlib_estatico(df, [])


### 6. Limpieza de nulos, negativos y días vacíos
 Convierte NaN y valores negativos a cero.
 Descarta los días completos cuya suma de radiación o
producción sea cero, eliminando jornadas sin ningún dato útil.


In [ ]:
df = limpiar_nulos_y_negativos(df)
df = eliminar_dias_vacios(df)


### 7. Corrección de radiación nocturna e inconsistencias sensor/producción
 Fuerza a cero cualquier valor de radiación en el
intervalo 22:00–06:00, eliminando lecturas físicamente imposibles del sensor.

 Emite una advertencia cuando detecta días con
radiación fija y positiva durante las 24 horas (síntoma de sensor bloqueado),
sin modificar los datos.

 Fuerza a cero la producción registrada en los
instantes en que la radiación es cero.


In [ ]:
df = corregir_radiacion_nocturna(df)
df = comprobar_radiacion_constante(df)
df = corregir_potencia_sin_radiacion(df)


### 8. Relleno de huecos temporales
 Reindexa el DataFrame para garantizar una frecuencia
estricta de 15 minutos y rellena los huecos en dos niveles:

1. **Interpolación lineal** para huecos de hasta 2 horas.
2. **Copia del mismo intervalo del día anterior** para huecos medianos.

Devuelve también  con las posiciones de los huecos detectados,
que se emplea en las visualizaciones posteriores.


In [ ]:
df, huecos_indices = procesar_huecos_temporales(df, rellenar=RELLENAR_HUECOS)
graficar_matplotlib_estatico(df, huecos_indices)


### 9. Detección y corrección de outliers
 Identifica valores extremos en potencia y radiación
mediante el método IQR y los corrige por clipping. Devuelve las listas de índices
con outliers detectados para su uso en el reporte interactivo.


In [ ]:
df, outliers_pow, outliers_rad = detectar_y_corregir_outliers(df)


### 10. Reporte interactivo y análisis de correlación mensual
 Produce un gráfico HTML que muestra los datos limpios
junto con los outliers corregidos y los huecos rellenados.
 Calcula y representa la correlación mensual entre
radiación y potencia para verificar la coherencia física del dataset.


In [ ]:
REPORT_PATH = str(RESULTS_DIR / "Analisis_Interactivo_LECA1.html")

fig_report = generar_reporte_interactivo(
    df, outliers_pow, outliers_rad, huecos_indices, REPORT_PATH
)
fig_report.show()


In [ ]:
_, fig_corr = analizar_correlacion_mensual(df)


### 11. Guardar dataset limpio
Se exporta el DataFrame procesado. Este fichero es la entrada para los pasos
de enriquecimiento con temperatura.


In [ ]:
df.to_csv(CSV_CLEAN, index=False)
print(f"Dataset limpio guardado en: {CSV_CLEAN}")
print(f"Registros: {len(df):,} | Periodo: {df['timestamp'].min()} -> {df['timestamp'].max()}")


### 12. Incorporación de temperatura ambiente
#### Opción A - Weather Underground
 Descarga (o recupera de caché) el histórico meteorológico
de las estaciones del pool.  Lo une con el dataset limpio de la
planta, alineando timestamps e interpolando la columna  a 15 minutos.

Esta opción se conserva como alternativa; la fuente preferida es Open-Meteo (opción B).


In [ ]:
STATION_IDS_WU = ["IALMAZ2", "IGOLMA6", "ISORIA35", "ICUBOD2"]

df_weather_wu = get_wunderground_history(
    start_date=datetime(2022, 1, 1),
    end_date=datetime(2024, 5, 31),
    station_ids=STATION_IDS_WU,
    cache_base_dir=WU_CACHE_DIR,
)

df_wu = process_and_merge(
    df_planta_path=CSV_CLEAN,
    df_weather=df_weather_wu,
    output_path=RESULTS_DIR / "Datos_LECA1_Con_Temperatura_WU.csv",
)


In [ ]:
# Reporte visual tras añadir temperatura WU
if df_wu is not None:
    fig_wu = generar_reporte_interactivo(
        df_wu, outliers_pow, outliers_rad, huecos_indices, REPORT_PATH
    )
    fig_wu.show()


#### Opción B - Open-Meteo (fuente preferida)
 Descarga el histórico horario de temperatura del
reanálisis ERA5 desde la API de Open-Meteo (sin clave de acceso), lo cachea
localmente y lo une al dataset limpio interpolando a 15 minutos.

Esta es la fuente preferida por su mayor cobertura temporal y consistencia.
El archivo generado es el punto de entrada del siguiente notebook.


In [ ]:
OPENMETEO_CACHE = PROJECT_ROOT / "data" / "external" / "openmeteo" / "openmeteo_raw.csv"
CSV_FINAL = RESULTS_DIR / "Datos_LECA1_Con_Temperatura_OpenMeteo.csv"

df = add_temperature_from_openmeteo(
    input_csv=CSV_CLEAN,
    output_csv=CSV_FINAL,
    openmeteo_cache_csv=OPENMETEO_CACHE,
    start_date="2022-01-01",
    end_date=datetime.now().strftime("%Y-%m-%d"),
)


In [ ]:
# Reporte visual tras añadir temperatura Open-Meteo
if df is not None:
    fig_om = generar_reporte_interactivo(
        df, outliers_pow, outliers_rad, huecos_indices, REPORT_PATH
    )
    fig_om.show()


### 13. Validación final del dataset
Se comprueba que el dataset final no contenga NaN ni valores infinitos en las
columnas críticas antes de pasar al siguiente notebook.


In [ ]:
critical_cols = ["power", "radiation", "T_ambiente"]

for col in critical_cols:
    n_nan = df[col].isna().sum()
    n_inf = np.isinf(df[col]).sum()
    print(f"{col:15s} | NaN: {n_nan:5d} | inf: {n_inf:5d} | min: {df[col].min():.3f} | max: {df[col].max():.3f}")


## Conclusiones del notebook 00

| Aspecto | Resultado |
|---|---|
| Periodo cubierto | Enero 2022 – Mayo 2024 |
| Resolución temporal | 15 minutos |
| Radiación rellena | Estaciones WU: ICUBOD2, ILOSRB1, IALMAZ2, IQUINT63, IGOLMA6, ISORIA35 |
| Anomalías corregidas | Sensor congelado, radiación nocturna, potencia sin radiación |
| Huecos temporales | Interpolados (<2 h) o copiados del día anterior |
| Outliers | Detectados por IQR y corregidos por clipping |
| Temperatura añadida | Open-Meteo (preferida) y Weather Underground (alternativa) |
 
**Punto de entrada del siguiente notebook:** análisis exploratorio y entrenamiento de modelos.
